In [ ]:
import json

import astropy.units as u
import gc_utils
import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from astropy.cosmology import Planck18 as cosmo
from astropy.cosmology import z_at_value
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from scipy.interpolate import interp1d

# plt.rc("font", family="Nimbus Roman")
plt.rc("font", family="Nimbus Roman No9 L")

SMALL_SIZE = 7
MEDIUM_SIZE = 7
BIGGER_SIZE = 7

plt.rc("font", size=SMALL_SIZE)  # controls default text sizes
plt.rc("axes", titlesize=SMALL_SIZE)  # fontsize of the axes title
plt.rc("axes", labelsize=MEDIUM_SIZE)  # fontsize of the x and y labels
plt.rc("xtick", labelsize=SMALL_SIZE)  # fontsize of the tick labels
plt.rc("ytick", labelsize=SMALL_SIZE)  # fontsize of the tick labels
plt.rc("legend", fontsize=SMALL_SIZE)  # legend fontsize
plt.rc("figure", titlesize=BIGGER_SIZE)  # fontsize of the figure title

In [ ]:
sim_lst = ["m12b", "m12c", "m12f", "m12i", "m12m"]
sim_dir = "/Users/z5114326/Documents/simulations/"

bands_dict = {"JC": ["v"], "SDSS": ["sdss_g", "sdss_z"]}

scmap = plt.get_cmap("tab10")
colors = [scmap(i) for i in np.linspace(0, 1, len(sim_lst))]
sim_colors = {sim: colors[i] for i, sim in enumerate(sim_lst)}

snap = 600
min_mass = 1e4

sim_halt_path = "../data/sim_data/sim_halt_dict.json"

In [ ]:
public_snapshot_file = sim_dir + "snapshot_times_public.txt"
pub_data = pd.read_table(public_snapshot_file, comment="#", header=None, sep=r"\s+")
pub_data.columns = [
    "index",
    "scale_factor",
    "redshift",
    "time_Gyr",
    "lookback_time_Gyr",
    "time_width_Myr",
]
timez0 = float(pub_data[pub_data["index"] == snap]["time_Gyr"].values[0])
pub_snaps = np.array(pub_data["index"])

all_data_fil = sim_dir + "/" + "m12i" + "/" + "m12i" + "_res7100/" + "snapshot_times.txt"
all_data = pd.read_table(all_data_fil, comment="#", header=None, sep=r"\s+")
all_data.columns = [
    "index",
    "scale_factor",
    "redshift",
    "time_Gyr",
    "lookback_time_Gyr",
    "time_width_Myr",
]
all_times = np.array(all_data["time_Gyr"])
all_snaps = np.array(all_data["index"])

snap_times = {
    gc_utils.snapshot_name(s): float(pub_data.loc[pub_data["index"] == s, "time_Gyr"].values[0])
    for s in pub_snaps
}

sim_codes = sim_dir + "simulation_codes.json"
with open(sim_codes) as sim_json:
    sim_data = json.load(sim_json)

dat_dir = "../data/"
mw_gcs_fil = "mw_gcs/mw_gcs.csv"
mw_gcs = pd.read_csv(dat_dir + mw_gcs_fil)

mw_gcs_cols_fil = "mw_gcs/mw_gcs_colors_ref.csv"
mw_gcs_cols = pd.read_csv(dat_dir + mw_gcs_cols_fil)

mw_gc_class = "../data/mw_gcs/class_chen.txt"
mw_class = pd.read_csv(mw_gc_class, sep=r"\s+")

mw_class.loc[mw_class["Cluster"] == "ESO-SC06", "Cluster"] = "ESO_280-SC06"
mw_class.loc[mw_class["Cluster"] == "1636-283", "Cluster"] = "ESO_452-SC11"

mw_class = mw_class.rename(columns={"Cluster": "cluster_name"})

to_path = "../data/sim_data/turn_offs_lbt.json"
with open(to_path, "r", encoding="utf-8") as f:
    to_dict = json.load(f)

In [ ]:
sim_dict = {}

# ------------------------------------------------
# precompute snapshot times once
# ------------------------------------------------
snap_times = {
    gc_utils.snapshot_name(s): float(pub_data.loc[pub_data["index"] == s, "time_Gyr"].values[0])
    for s in pub_snaps
}

timez0 = snap_times[gc_utils.snapshot_name(snap)]

for sim in sim_lst:
    ghost_file = f"{sim_dir}{sim}/{sim}_ghosts.hdf5"
    sim_dict[sim] = {}

    with h5py.File(ghost_file, "r") as ghost_data:
        for it_id, it_grp in ghost_data.items():
            src = it_grp["source"]

            # ---------------------------
            # read once
            # ---------------------------
            grpid = src["grpid"][()]
            amsk = grpid == 0

            logm_tfor = src["logm_tfor"][()]
            logm_tz0 = src["logm_tz0"][()]

            m_tfo = 10.0**logm_tfor
            m_tfo_ev = 0.55 * m_tfo  # stellar-evolution corrected mass

            m_tz0 = np.zeros_like(logm_tz0, dtype=float)
            msk = logm_tz0 != -1
            m_tz0[msk] = 10.0 ** logm_tz0[msk]

            tfor = src["tfor"][()]
            tdis = src["tdis"][()]
            tacc = src["tacc"][()]
            feh = src["feh"][()]
            gcids = src["gcid"][()]

            tdis = np.where(tdis == -1, np.inf, tdis)

            # ---------------------------
            # snapshot mass table
            #   + tfor (m_tfo_ev)
            #   + tdis (0)
            # ---------------------------
            n_gc = len(gcids)
            n_snap = len(pub_snaps)

            mass_arr = np.full((n_gc, n_snap + 2), np.nan)
            time_arr = np.full((n_gc, n_snap + 2), np.nan)

            # ---- formation point ----
            mass_arr[:, 0] = m_tfo_ev
            time_arr[:, 0] = tfor

            # ---- simulation snapshots ----
            for j, (snap_id, time) in enumerate(snap_times.items(), start=1):
                snp = it_grp["snapshots"][snap_id]
                snp_gcids = snp["gcid"][()]
                snp_mass = 10.0 ** snp["logm"][()]

                common, idx_snp, idx_src = np.intersect1d(snp_gcids, gcids, return_indices=True)

                mass_arr[idx_src, j] = snp_mass[idx_snp]
                time_arr[idx_src, j] = time

                # ---- disruption point (only if disrupted) ----
                has_disrupted = np.isfinite(tdis)

                mass_arr[has_disrupted, -1] = 0.0
                time_arr[has_disrupted, -1] = tdis[has_disrupted]

            # ---------------------------
            # final mass cut
            # ---------------------------
            m_tfo_msk = m_tfo >= min_mass

            sim_dict[sim][it_id] = {
                "gcid": gcids[m_tfo_msk],
                "feh": feh[m_tfo_msk],
                "tfor": tfor[m_tfo_msk],
                "tdis": tdis[m_tfo_msk],
                "tacc": tacc[m_tfo_msk],
                "acc": amsk[m_tfo_msk],
                # ---- time-resolved mass history ----
                "m_snp": mass_arr[m_tfo_msk],
                "t_snp": time_arr[m_tfo_msk],
            }

In [ ]:
def build_mass_interpolators(dat):
    interps = []

    for m, t in zip(dat["m_snp"], dat["t_snp"]):
        msk = np.isfinite(m) & (m > 0)
        if np.sum(msk) < 2:
            interps.append(None)
            continue

        order = np.argsort(t[msk])
        ts = t[msk][order]
        ms = m[msk][order]

        interps.append(interp1d(ts, ms, kind="linear", bounds_error=False, fill_value=(ms[0], 0.0)))

    return interps


for sim in sim_lst:
    for it_id in sim_dict[sim]:
        sim_dict[sim][it_id]["mass_interp"] = build_mass_interpolators(sim_dict[sim][it_id])

In [ ]:
snaps = [120, 277, 382, 600]  # z=3, 1, 0.5, 0
tims = np.array([pub_data[pub_data["index"] == snap]["time_Gyr"].values[0] for snap in snaps])
zs = np.round(np.array([pub_data[pub_data["index"] == snap]["redshift"].values[0] for snap in snaps]), 1)

In [ ]:
# # ------------------------------------------------------------------
# # time grid
# dt = 0.2
# tims = np.arange(0.4, timez0, dt)
# tims = np.append(tims, timez0)


def quant_age(age, dt=0.2):
    """Quantise age to SSP time grid to avoid float lookup issues."""
    return np.round(age / dt) * dt


# ------------------------------------------------------------------
for sim in sim_lst:
    print(f"[SIM] {sim}")

    for band_type in bands_dict.keys():
        # ---------- band info ----------
        band_names = bands_dict[band_type]
        n_band = len(band_names)
        bands_tag = "_".join(band_names)

        # ---------- load SSP cache ----------
        cache_dir = f"../data/ssp_caches/{sim}"
        cache_path = f"{cache_dir}/ssp_{band_type}_{bands_tag}.npz"

        cache = np.load(cache_path)
        ssp_mag = cache["ssp_mag"]  # shape: (n_ssp, n_band)
        ssp_mass = cache["ssp_mass"]  # shape: (n_ssp,)
        unique_ssps = cache["unique_ssps"]  # shape: (n_ssp, 2) [age, feh]

        # ---------- build SSP lookup ----------
        ssp_lookup = {(quant_age(unique_ssps[i, 0]), unique_ssps[i, 1]): i for i in range(len(unique_ssps))}

        # ---------- loop over iterations ----------
        for it_id, dat in sim_dict[sim].items():
            if it_id == "ssp_times":
                continue

            print(f"    {band_type}  -  {it_id}")

            gcids = dat["gcid"]
            tfor = dat["tfor"]
            feh = dat["feh"]
            tdis = dat["tdis"]
            tacc = dat["tacc"]
            mass_interp = dat["mass_interp"]

            n_gc = len(gcids)
            n_t = len(tims)

            # ---------- output magnitude cube ----------
            mag_cube = np.full((n_gc, n_t, n_band), np.nan, dtype=float)

            # ---------- time evolution ----------
            for ti, t in enumerate(tims):
                alive = (tfor <= t) & (t < tdis) & (tacc <= t)

                for i in np.where(alive)[0]:
                    fmass = mass_interp[i]
                    if fmass is None:
                        continue

                    m_now = float(fmass(t))
                    if m_now <= 0:
                        continue

                    age = t - tfor[i]
                    if age <= 0:
                        continue

                    key = (quant_age(age), feh[i])
                    ssp_idx = ssp_lookup.get(key)

                    if ssp_idx is None:
                        continue

                    mag_cube[i, ti, :] = ssp_mag[ssp_idx] - 2.5 * np.log10(m_now / ssp_mass[ssp_idx])

            # ---------- store per-band results ----------
            for b, band_name in enumerate(band_names):
                dat[band_name] = mag_cube[:, :, b]

In [ ]:
def mags_hist(
    ax,
    sim,
    sim_dict,
    tim,
    tims,
    bins,
    bands=["v"],
    origin=False,
    plt_text=False,
):
    ax.clear()

    if len(bands) == 1:
        colors = False
        print(f"Plotting {bands[0]} magnitudes")
    elif len(bands) == 2:
        colors = True
        print(f"Plotting {bands[0]}-{bands[1]} color")
    else:
        raise RuntimeError("Can't do origin and gauss_fit at the same time, choose one")

    hist_dict = {"sum": None, "insum": None, "exsum": None, "n": 0}

    # tims = sim_dict["times"]
    tidx = np.where(tims == tim)[0][0]

    all_mags = []
    for it_id in sim_dict[sim].keys():
        insitu_msk = sim_dict[sim][it_id]["acc"]
        taccs = sim_dict[sim][it_id]["tacc"]
        tacc_msk = tim > taccs

        if colors:
            mag0 = sim_dict[sim][it_id][bands[0]][:, tidx]
            mag1 = sim_dict[sim][it_id][bands[1]][:, tidx]
            mags_hld = mag0 - mag1
        else:
            mags_hld = sim_dict[sim][it_id][bands[0]][:, tidx]

        fin_msk = np.isfinite(mags_hld)

        mags = mags_hld[fin_msk & tacc_msk]
        mags_in = mags_hld[fin_msk & tacc_msk & insitu_msk]
        mags_ex = mags_hld[fin_msk & tacc_msk & ~insitu_msk]

        all_mags.append(mags)

        hssp, _ = np.histogram(mags, bins=bins)
        hssp_in, _ = np.histogram(mags_in, bins=bins)
        hssp_ex, _ = np.histogram(mags_ex, bins=bins)

        if hist_dict["sum"] is None:
            hist_dict["sum"] = hssp.astype(float)
            hist_dict["insum"] = hssp_in.astype(float)
            hist_dict["exsum"] = hssp_ex.astype(float)
        else:
            hist_dict["sum"] += hssp
            hist_dict["insum"] += hssp_in
            hist_dict["exsum"] += hssp_ex

        hist_dict["n"] += 1

    binsum = hist_dict["sum"] / hist_dict["n"]
    binsum_in = hist_dict["insum"] / hist_dict["n"]
    binsum_ex = hist_dict["exsum"] / hist_dict["n"]

    if origin:
        ax.step(bins[:-1], binsum_in, where="post", color="r", label="in situ", lw=0.5)
        ax.step(bins[:-1], binsum_ex, where="post", color="b", label="ex situ", lw=0.5)

    ax.step(bins[:-1], binsum, where="post", color="k", linewidth=1, label="total")

    if plt_text:
        if tim > 13.78:
            z = 0
        else:
            z = z_at_value(cosmo.age, tim * u.Gyr)

        text = (
            sim + "\n" + "t = " + str(np.round(tim, 2)) + " Gyr" + "\n" + "z = " + str(np.round(float(z), 2))
        )

        ax.text(
            0.02,
            0.92,
            text,
            transform=ax.transAxes,
            color="k",
            ha="left",
            va="center",
            bbox=dict(
                facecolor="white",
                edgecolor="white",
            ),
        )

        ax.legend()

    ax.set_ylabel("Count")


def plot_mw_mags_hist(
    ax,
    mw_gcs,
    mw_gcs_cols,
    mw_class,
    bins,
    plot_type="mag",
    origin=False,
    plt_text=False,
):

    if plot_type not in ("mag", "col"):
        raise RuntimeError("plot_type must be 'mag' or 'col'")

    # Ensure cluster_name is string everywhere
    for df in (mw_class, mw_gcs, mw_gcs_cols):
        df["cluster_name"] = df["cluster_name"].astype(str)

    # Merge Origin from mw_class with V_abs from mw_gcs
    df = mw_class.merge(
        mw_gcs[["cluster_name", "V_abs"]],
        on="cluster_name",
        how="inner",
    )

    # Merge colour columns from mw_gcs_cols
    df = df.merge(
        mw_gcs_cols[
            [
                "cluster_name",
                "JC_V-SDSS_g",
                "JC_V-SDSS_z",
            ]
        ],
        on="cluster_name",
        how="inner",
    )

    mw_v = df["V_abs"]
    mw_g = mw_v - df["JC_V-SDSS_g"]
    mw_z = mw_v - df["JC_V-SDSS_z"]
    inmsk = df["Origin"] == "in-situ"

    if plot_type == "mag":
        mag = mw_v
        xlab = r"M$_{V}$"
    elif plot_type == "col":
        mag = mw_g - mw_z
        xlab = r"(g-z)$_{0}$"

    if origin:
        ax.hist(mag[inmsk], bins=bins, color="r", label="in situ", histtype="step", lw=0.5)
        ax.hist(mag[~inmsk], bins=bins, color="b", label="ex situ", histtype="step", lw=0.5)

    ax.hist(mag, bins=bins, color="k", histtype="step", lw=1, label="total")

    if plt_text:
        ax.legend()

        text = "Milky Way" + "\n" + "t = " + str(13.80) + " Gyr" + "\n" + "z = " + str(0)

        ax.text(
            0.02,
            0.92,
            text,
            transform=ax.transAxes,
            color="k",
            ha="left",
            va="center",
            bbox=dict(
                facecolor="white",
                edgecolor="white",
            ),
        )

    ax.set_xlim(bins[0], bins[-1])
    ax.set_ylim(0, 50)

    ax.set_xlabel(xlab)
    ax.set_ylabel("Count")


def plot_hist(ax, sim, sim_dict, tim, bins, prop="feh"):

    hist_dict = {"sum": None, "insum": None, "exsum": None, "n": 0}

    for it_id in sim_dict[sim].keys():
        inmsk = sim_dict[sim][it_id]["acc"]

        taccs = sim_dict[sim][it_id]["tacc"]
        tfors = sim_dict[sim][it_id]["tfor"]
        tdiss = sim_dict[sim][it_id]["tdis"]

        tmsk = (taccs <= tim) & (tfors <= tim) & (tim < tdiss)

        if prop == "feh":
            # prop_i = sim_dict[sim][it_id]["feh"][tidx]
            prop_al = sim_dict[sim][it_id]["feh"][tmsk]
            prop_in = sim_dict[sim][it_id]["feh"][tmsk & inmsk]
            prop_ex = sim_dict[sim][it_id]["feh"][tmsk & ~inmsk]

        elif prop == "age":
            prop_al = tim - sim_dict[sim][it_id]["tfor"][tmsk]
            prop_in = tim - sim_dict[sim][it_id]["tfor"][tmsk & inmsk]
            prop_ex = tim - sim_dict[sim][it_id]["tfor"][tmsk & ~inmsk]

        hssp, _ = np.histogram(prop_al, bins=bins)
        hssp_in, _ = np.histogram(prop_in, bins=bins)
        hssp_ex, _ = np.histogram(prop_ex, bins=bins)

        if hist_dict["sum"] is None:
            hist_dict["sum"] = hssp.astype(float)
            hist_dict["insum"] = hssp_in.astype(float)
            hist_dict["exsum"] = hssp_ex.astype(float)
        else:
            hist_dict["sum"] += hssp
            hist_dict["insum"] += hssp_in
            hist_dict["exsum"] += hssp_ex

        hist_dict["n"] += 1

    binsum = hist_dict["sum"] / hist_dict["n"]
    binsum_in = hist_dict["insum"] / hist_dict["n"]
    binsum_ex = hist_dict["exsum"] / hist_dict["n"]

    ax.step(bins[:-1], binsum, where="post", color="k", linewidth=1, label="total")
    ax.step(bins[:-1], binsum_in, where="post", color="r", label="in situ", lw=0.5)
    ax.step(bins[:-1], binsum_ex, where="post", color="b", label="ex situ", lw=0.5)


def gc_masses_at_time(tim, select, t_snp, m_snp):
    """
    Return array of GC masses at time tim.
    """
    masses = []

    for i in np.where(select)[0]:
        x = t_snp[i]
        y = m_snp[i]

        mask = np.isfinite(x) & np.isfinite(y)
        if mask.sum() >= 2:
            masses.append(np.interp(tim, x[mask], y[mask]))

    return np.asarray(masses)


def mass_histogram(ax, sim, sim_dict, tim, density=False):
    ax.clear()
    bins = 10 ** np.linspace(2, 7.1, 52)

    hist_dict = {"insum": None, "exsum": None, "n": 0}

    for it_id in sim_dict[sim].keys():
        dat = sim_dict[sim][it_id]

        tfor = dat["tfor"]
        tdis = dat["tdis"].copy()
        tdis[tdis == -1] = np.inf
        tacc = dat["tacc"]

        t_snp = dat["t_snp"]
        m_snp = dat["m_snp"]

        alive = (tfor <= tim) & (tim < tdis)
        inmw = tacc <= tim
        insitu = dat["acc"]

        in_select = alive & inmw & insitu
        ex_select = alive & inmw & ~insitu

        in_masses = gc_masses_at_time(tim, in_select, t_snp, m_snp)
        ex_masses = gc_masses_at_time(tim, ex_select, t_snp, m_snp)

        hin, _ = np.histogram(in_masses, bins=bins)
        hex, _ = np.histogram(ex_masses, bins=bins)

        if hist_dict["insum"] is None:
            hist_dict["insum"] = hin.astype(float)
            hist_dict["exsum"] = hex.astype(float)
        else:
            hist_dict["insum"] += hin
            hist_dict["exsum"] += hex

        hist_dict["n"] += 1

    # bin_centers = 0.5 * (bins[:-1] + bins[1:])
    widths = bins[1:] - bins[:-1]

    insum = hist_dict["insum"] / hist_dict["n"]
    exsum = hist_dict["exsum"] / hist_dict["n"]

    if density:
        insum = insum / np.sum(insum * widths)
        exsum = exsum / np.sum(exsum * widths)

    ax.step(bins[:-1], insum, where="post", color="r", label="in situ", lw=0.5)
    # ax.fill_between(bins[:-1], insum, step="post", alpha=0.5, color="r")

    ax.step(bins[:-1], exsum, where="post", color="b", label="ex situ", lw=0.5)
    # ax.fill_between(bins[:-1], exsum, step="post", alpha=0.5, color="b")

    ax.step(bins[:-1], insum + exsum, where="post", color="k", label="total", lw=1)

    ax.set_xscale("log")
    ax.set_xlabel(r"M$_{\rm GC}$ [M$_\odot$]")
    ax.set_ylabel("Count" if not density else "PDF")
    # ax.legend()

    ax.set_xlim(10**2, 10**7)
    ax.set_ylim(0)

In [ ]:
# MW Overlay ####################################################
# Ensure cluster_name is string everywhere
for df in (mw_class, mw_gcs, mw_gcs_cols):
    df["cluster_name"] = df["cluster_name"].astype(str)

# Merge Origin from mw_class with V_abs from mw_gcs
df = mw_class.merge(
    mw_gcs[["cluster_name", "V_abs", "feh", "mass"]],
    on="cluster_name",
    how="inner",
)

# Merge colour columns from mw_gcs_cols
df = df.merge(
    mw_gcs_cols[
        [
            "cluster_name",
            "JC_V-SDSS_g",
            "JC_V-SDSS_z",
        ]
    ],
    on="cluster_name",
    how="inner",
)

mw_v = df["V_abs"]
mw_g = mw_v - df["JC_V-SDSS_g"]
mw_z = mw_v - df["JC_V-SDSS_z"]

mw_col = mw_g - mw_z

mw_mas = df["mass"]

In [ ]:
mag_bins = np.arange(-12, -0.5, 0.5)
col_bins = np.arange(-1.0, 2.1, 0.1)
feh_bins = np.arange(-3.6, 1.6, 0.2)
age_bins = np.arange(0, 14, 0.2)

sim_leg = [
    Line2D([0], [0], color="red", lw=1.5, label="in situ"),
    Line2D([0], [0], color="blue", lw=1.5, label="ex situ"),
    Line2D([0], [0], color="black", lw=1.5, label="total"),
]

mw_leg = [Patch(facecolor="magenta", edgecolor="magenta", alpha=0.2, label="Milky Way")]

# Magnitudes and Colours

In [ ]:
ylim_mag_dict = {
    "m12b": [500, 500, 150, 80],
    "m12c": [500, 1250, 300, 70],
    "m12f": [1000, 500, 250, 80],
    "m12i": [420, 1500, 250, 60],
    "m12m": [170, 1500, 400, 100],
}

for sim in sim_lst:
    fig, axs = plt.subplots(nrows=4, ncols=3, figsize=(7, 4.67))
    plt.subplots_adjust(hspace=0.12, wspace=0.075)

    for (i, tim), z in zip(enumerate(tims), zs):
        mags_hist(axs[i, 0], sim, sim_dict, tim, tims, mag_bins, bands=["v"], origin=True)
        axs[i, 0].set_xlim(0.5, -12.5)

        mags_hist(axs[i, 1], sim, sim_dict, tim, tims, col_bins, bands=["sdss_g", "sdss_z"], origin=True)
        axs[i, 1].set_xlim(-0.8, 1.8)

        plot_hist(axs[i, 2], sim, sim_dict, tim, feh_bins, prop="feh")

        if i == 3:
            axs[i, 0].set_xlabel(r"M$_{\mathrm{V}}$ (Mag)")
            axs[i, 1].set_xlabel(r"(g-z)$_{0}$ (Mag)")
            axs[i, 2].set_xlabel(r"[Fe/H]")
        else:
            axs[i, 0].set_xticklabels([])
            axs[i, 1].set_xticklabels([])
            axs[i, 2].set_xticklabels([])

        axs[i, 0].set_ylim(0, ylim_mag_dict[sim][i])
        axs[i, 1].set_ylim(0, ylim_mag_dict[sim][i])
        axs[i, 2].set_ylim(0, ylim_mag_dict[sim][i])

        axs[i, 1].set_yticklabels([])
        axs[i, 2].set_yticklabels([])

        axs[i, 1].set_ylabel("")

        tlb = timez0 - tim
        x = 0.03
        y = 0.95

        axs[i, 0].text(x, y, r"t$_{\mathrm{lb}}$", transform=axs[i, 0].transAxes, ha="left", va="top")
        axs[i, 0].text(x + 0.04, y, rf"$= {tlb:.1f}$ Gyr", transform=axs[i, 0].transAxes, ha="left", va="top")

        axs[i, 0].text(x + 0.01, y - 0.15, r"$z$", transform=axs[i, 0].transAxes, ha="left", va="top")
        axs[i, 0].text(
            x + 0.04, y - 0.15, rf"$= {z:.1f}$", transform=axs[i, 0].transAxes, ha="left", va="top"
        )

    axs[3, 0].hist(mw_v, color="magenta", bins=mag_bins, alpha=0.2)
    axs[3, 1].hist(mw_col, color="magenta", bins=col_bins, alpha=0.2)
    axs[3, 2].hist(df["feh"], color="magenta", bins=feh_bins, alpha=0.2)

    axs[0, 2].legend(handles=sim_leg, loc="upper right", ncol=1, frameon=False)
    axs[3, 2].legend(handles=mw_leg, loc="upper right", ncol=1, frameon=False)

    save_name = "../data/figures/gc_plots/" + sim + "_mags.pdf"
    fig.savefig(save_name, bbox_inches="tight")


In [ ]:
len(mw_v), len(mw_col), len(df["feh"])

In [ ]:
mw_gcs.keys()

In [ ]:
mw_gcs["cluster_name"]

In [ ]:
mw_gcs["age_gyr"]

# Masses

In [ ]:
# ylim_mas_dict = {
#     "m12b": [300, 150, 60, 30],
#     "m12c": [350, 300, 100, 30],
#     "m12f": [500, 150, 75, 30],
#     "m12i": [300, 400, 100, 20],
#     "m12m": [100, 300, 150, 40],
# }

# mas_bins = 10 ** np.linspace(2, 7.1, 52)

# for sim in sim_lst:
#     fig, axs = plt.subplots(nrows=4, ncols=1, figsize=(5, 10))
#     plt.subplots_adjust(hspace=0.1, wspace=0.05)

#     for (i, tim), z in zip(enumerate(tims), zs):
#         mass_histogram(axs[i], sim, sim_dict, tim, density=False)

#         if i != 3:
#             axs[i].set_xlabel("")
#             axs[i].set_xticklabels([])

#         axs[i].set_ylim(0, ylim_mas_dict[sim][i])

#         tlb = timez0 - tim
#         x = 0.03
#         y = 0.95

#         axs[i].text(x, y, r"t$_{\mathrm{lb}}$", transform=axs[i].transAxes, ha="left", va="top")
#         axs[i].text(x + 0.04, y, rf"$= {tlb:.1f}$ Gyr", transform=axs[i].transAxes, ha="left", va="top")

#         axs[i].text(x + 0.01, y - 0.10, r"$z$", transform=axs[i].transAxes, ha="left", va="top")
#         axs[i].text(x + 0.04, y - 0.10, rf"$= {z:.1f}$", transform=axs[i].transAxes, ha="left", va="top")

#     axs[3].hist(df["mass"], color="magenta", bins=mas_bins, alpha=0.2)

#     axs[0].legend(handles=sim_leg, loc="upper right", ncol=1, frameon=False)
#     axs[3].legend(handles=mw_leg, loc="upper right", ncol=1, frameon=False)

#     save_name = "../data/figures/gc_plots/" + sim + "_mass.pdf"
#     fig.savefig(save_name, bbox_inches="tight")

# Normalised z=0 comparisons

In [ ]:
def normalise_cols(sim, ghost_data, col_bins):

    cache_dir = f"../data/ssp_caches/{sim}"
    cache_path = f"{cache_dir}/ssp_{band_type}_{bands_tag}.npz"

    cache = np.load(cache_path)
    ssp_mag = cache["ssp_mag"]  # shape: (n_ssp, n_band)
    unique_ssps = cache["unique_ssps"]  # shape: (n_ssp, 2) [age, feh]

    ssp_lookup = {(quant_age(unique_ssps[i, 0]), unique_ssps[i, 1]): i for i in range(len(unique_ssps))}

    n = 0

    hs_sum = 0
    hs_in_sum = 0
    hs_ex_sum = 0

    hd_sum = 0
    hd_in_sum = 0
    hd_ex_sum = 0

    for it_id in ghost_data.keys():
        src_dat = ghost_data[it_id]["source"]
        smsk = np.array(src_dat["s_flag"][()], dtype=bool)
        amsk = src_dat["grpid"][()] == 0

        fehs = src_dat["feh"][()]
        ages = timez0 - src_dat["tfor"][()]

        cols = []
        for i in range(len(smsk)):
            key = (quant_age(ages[i]), fehs[i])
            ssp_idx = ssp_lookup.get(key)
            mags_i = ssp_mag[ssp_idx]
            col_i = mags_i[0] - mags_i[1]
            cols.append(col_i)
        cols = np.array(cols)

        fmsk = np.isfinite(cols)

        hs, _ = np.histogram(cols[fmsk & smsk], bins=col_bins)
        hs_in, _ = np.histogram(cols[fmsk & smsk & amsk], bins=col_bins)
        hs_ex, _ = np.histogram(cols[fmsk & smsk & ~amsk], bins=col_bins)

        hd, _ = np.histogram(cols[fmsk & ~smsk], bins=col_bins)
        hd_in, _ = np.histogram(cols[fmsk & ~smsk & amsk], bins=col_bins)
        hd_ex, _ = np.histogram(cols[fmsk & ~smsk & ~amsk], bins=col_bins)

        hs_sum += hs
        hs_in_sum += hs_in
        hs_ex_sum += hs_ex

        hd_sum += hd
        hd_in_sum += hd_in
        hd_ex_sum += hd_ex

        n += 1

    hs_sum = hs_sum / n
    hs_in_sum = hs_in_sum / n
    hs_ex_sum = hs_ex_sum / n

    hd_sum = hd_sum / n
    hd_in_sum = hd_in_sum / n
    hd_ex_sum = hd_ex_sum / n

    widths = col_bins[1:] - col_bins[:-1]
    hs_norm = np.sum(hs_sum * widths)
    hd_norm = np.sum(hd_sum * widths)

    hs_den = hs_sum / hs_norm
    hs_in_den = hs_in_sum / hs_norm
    hs_ex_den = hs_ex_sum / hs_norm

    hd_den = hd_sum / hd_norm
    hd_in_den = hd_in_sum / hd_norm
    hd_ex_den = hd_ex_sum / hd_norm

    return hs_den, hs_in_den, hs_ex_den, hd_den, hd_in_den, hd_ex_den

In [ ]:
# col_bins = np.arange(0, 1.9, 0.1)

# band_type = "SDSS"
# band_names = bands_dict[band_type]
# n_band = len(band_names)
# bands_tag = "_".join(band_names)

# fig, axs = plt.subplots(nrows=2, ncols=3, figsize=(15, 8))
# plt.subplots_adjust(hspace=0.05, wspace=0.05)

# for i, ax in enumerate(axs.flatten()):
#     if i > 4:
#         ax.axis("off")
#         continue

#     sim = sim_lst[i]
#     print(sim)

#     ghost_file = f"{sim_dir}{sim}/{sim}_ghosts.hdf5"
#     ghost_data = h5py.File(ghost_file, "r")  # open processed data file

#     hs_den, hs_in_den, hs_ex_den, _, hd_in_den, hd_ex_den = normalise_cols(sim, ghost_data, col_bins)

#     ax.step(col_bins[:-1], hs_den, where="post", color="k", lw=2, label="all")
#     ax.step(col_bins[:-1], hs_in_den, where="post", color="r", lw=1, label="in-situ")
#     ax.step(col_bins[:-1], hs_ex_den, where="post", color="b", lw=1, label="ex-situ")

#     ax.fill_between(col_bins[:-1], 0, hd_in_den, step="post", color="r", alpha=0.15)
#     ax.fill_between(col_bins[:-1], 0, hd_ex_den, step="post", color="b", alpha=0.15)

#     ax.set_xlim(0.3, 1.7)
#     ax.set_ylim(0, 3.1)

#     if i < 2:
#         ax.set_xticklabels([])
#     else:
#         ax.set_xlabel(r"(g-z)$_{0}$ [Mag]")

#     if i not in [0, 3]:
#         ax.set_yticklabels([])
#     else:
#         ax.set_ylabel(r"Density")

#     ax.text(0.85, 0.90, sim, ha="left", va="center", transform=ax.transAxes)

#     if i == 0:
#         ax.legend(loc="upper left")

# save_name = "../data/figures/gc_plots/z0_colour_comparison.pdf"
# fig.savefig(save_name, bbox_inches="tight")

In [ ]:
col_bins = np.arange(0, 1.9, 0.1)

band_type = "SDSS"
band_names = bands_dict[band_type]
n_band = len(band_names)
bands_tag = "_".join(band_names)

fig, axs = plt.subplots(3, 2, figsize=(3.5, 3.75))
plt.subplots_adjust(hspace=0.085, wspace=0.085)

for i, ax in enumerate(axs.flatten()):
    if i > 4:
        ax.axis("off")

        handles, labels = axs[0, 0].get_legend_handles_labels()
        ax.legend(handles, labels, loc="lower left", frameon=True)

        continue

    sim = sim_lst[i]
    print(sim)

    ghost_file = f"{sim_dir}{sim}/{sim}_ghosts.hdf5"
    ghost_data = h5py.File(ghost_file, "r")  # open processed data file

    hs_den, hs_in_den, hs_ex_den, _, hd_in_den, hd_ex_den = normalise_cols(sim, ghost_data, col_bins)

    ax.step(col_bins[:-1], hs_in_den, where="post", color="r", lw=0.5, label="in situ")
    ax.step(col_bins[:-1], hs_ex_den, where="post", color="b", lw=0.5, label="ex situ")
    ax.step(col_bins[:-1], hs_den, where="post", color="k", lw=1, label="total")

    ax.fill_between(col_bins[:-1], 0, hd_in_den, step="post", color="r", alpha=0.15)
    ax.fill_between(col_bins[:-1], 0, hd_ex_den, step="post", color="b", alpha=0.15)

    ax.set_xlim(0.3, 1.7)
    ax.set_ylim(0, 3.3)

    if i < 3:
        ax.set_xticklabels([])
    else:
        ax.set_xlabel(r"(g-z)$_{0}$ (Mag)")

    if i in [0, 2, 4]:
        ax.set_ylabel(r"Density")
    else:
        ax.set_yticklabels([])

    ax.text(0.75, 0.85, sim, ha="left", va="center", transform=ax.transAxes)

    # if i == 0:
    #     ax.legend(loc="upper left")

save_name = "../data/figures/gc_plots/z0_colour_comparison.pdf"
fig.savefig(save_name, bbox_inches="tight")

# Ratios 

In [ ]:
ratio_dict = {sim: {"in": None, "ex": None} for sim in sim_lst}
tims = np.arange(1, 14, 0.1)

for sim in sim_lst:
    incnt = []
    excnt = []
    total = []
    for it_id in sim_dict[sim].keys():
        dat = sim_dict[sim][it_id]

        tfor = dat["tfor"]
        tdis = dat["tdis"].copy()
        tdis[tdis == -1] = np.inf
        tacc = dat["tacc"]

        t_snp = dat["t_snp"]
        m_snp = dat["m_snp"]

        incnt_i = []
        excnt_i = []
        total_i = []
        for tim in tims:
            alive = (tfor <= tim) & (tim < tdis)
            inmw = tacc <= tim
            insitu = dat["acc"]

            incnt_i.append(np.sum(alive & inmw & insitu))
            excnt_i.append(np.sum(alive & inmw & ~insitu))
            total_i.append(np.sum(alive & inmw))

        incnt.append(np.asarray(incnt_i))
        excnt.append(np.asarray(excnt_i))
        total.append(np.asarray(total_i))

    inrat = []
    exrat = []
    for i, _ in enumerate(incnt):
        inrat.append(incnt[i] / total[i])
        exrat.append(excnt[i] / total[i])

    inrat_avg = np.mean(inrat, axis=0)
    exrat_avg = np.mean(exrat, axis=0)

    ratio_dict[sim]["in"] = inrat_avg
    ratio_dict[sim]["ex"] = exrat_avg

In [ ]:
fig, axs = plt.subplots(nrows=2, ncols=3, figsize=(16, 8))
plt.subplots_adjust(hspace=0.075, wspace=0.075)

include_mw = True
mw_handle = Line2D([0], [0], marker="*", color="grey", linestyle="None", markersize=7, label="Milky Way")

for s, ax in enumerate(axs.flatten()):
    if s > len(sim_lst) - 1:
        ax.axis("off")
        continue
    sim = sim_lst[s]

    ax.plot(timez0 - tims, ratio_dict[sim]["in"], c="r", label="In-situ")
    ax.plot(timez0 - tims, ratio_dict[sim]["ex"], c="b", label="Ex-situ")

    ax.set_xlim(timez0 - 1, -0.25)
    ax.set_ylim(0, 1)

    if s < 2:
        ax.set_xticklabels([])
    else:
        ax.set_xlabel("Lookback Time (Gyr)")

    if s in [1, 2, 4]:
        ax.set_yticklabels([])
    else:
        ax.set_ylabel("Fraction of Total Clusters")

    ax.axvline(to_dict["cnt"][sim], ls=":", c="k")

    ax.text(0.87, 0.92, sim, transform=ax.transAxes, color="k", ha="left")

    if include_mw:
        unq_class, cnts_class = np.unique(mw_class["Origin"], return_counts=True)

        inmsk = unq_class == "in-situ"

        in_sum = np.sum(cnts_class[inmsk])
        ex_sum = np.sum(cnts_class[~inmsk])

        ax.scatter(0, in_sum / (in_sum + ex_sum), marker="*", c="r", s=80, zorder=10)
        ax.scatter(0, ex_sum / (in_sum + ex_sum), marker="*", c="b", s=80, zorder=10)

    if s == 0:
        handles, labels = ax.get_legend_handles_labels()
        handles.append(mw_handle)
        labels.append("Milky Way")
        ax.legend(handles, labels, frameon=False, loc="lower right")

save_name = "../data/figures/gc_plots/cluster_ratios.pdf"
fig.savefig(save_name, bbox_inches="tight")